# Reproducing the experiment in Google Colab

API keys are read from Colab **Secrets** (key icon in the left bar): `ANTHROPIC_API_KEY` and `OPENAI_API_KEY`. Never paste keys into a cell.

In [ ]:
!git clone https://github.com/Smil1e/llm-ambiguity-user-stories.git
%cd llm-ambiguity-user-stories
!pip -q install -r requirements.txt
!python -m spacy download en_core_web_sm -q

## 1. Re-evaluate the saved outputs (no API calls)
Reproduces Tables 4-7 and Appendix A of the report from `results/full/raw_*.jsonl`.

In [ ]:
!python evaluate.py --set full
!python analyze_originals.py --flags results/full/flagged_originals.csv --stories data/base_stories.csv --out results/full
!python sensitivity.py

## 2. Optional: rebuild the stories and rerun the models (costs API credit)
Download the Dalpiaz user-story data set (doi:10.17632/7zbk8zsd8y), put its `.txt` files in a folder, and set `DATA` below. `prepare.py` is deterministic (seed 42).

In [ ]:
import os
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY').strip()
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY').strip()
os.environ['CLAUDE_MODEL'] = 'claude-haiku-5-5'
os.environ['OPENAI_MODEL'] = 'gpt-6-luna'
DATA = '/content/dalpiaz'  # folder with the data set files

In [ ]:
!python prepare.py --data "{DATA}" --pool-all --fewshot-project g21-badcamp
!python check_grammar.py --set full
!python run_experiment.py --set full --models rule claude openai --strategies zero few --runs 2
!python evaluate.py --set full